In [1]:
!pip install -U --quiet fhir.resources langgraph langchain-google-genai langchain-community langchain-core faiss-cpu sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.


In [2]:
# ==============================================================================
# EVALUACIÓN PARCIAL N°2 — AGENTE CLÍNICO ADAPTATIVO "NEXUS-V VITALIS"
# ==============================================================================
# Asignatura: Ingeniería de Soluciones con IA (ISY0101)
# Frameworks: LangGraph, LangChain, Pydantic v2, FHIR Resources, Google Gemini
# ==============================================================================

import os
import json
import time
import operator
import warnings
from typing import Annotated, TypedDict, List, Dict, Any
from pydantic import BaseModel, Field
from IPython.display import display, Markdown  # Permite renderizado visual en Colab/Jupyter

# Silenciar advertencias de deprecación menores para mantener la consola limpia
warnings.filterwarnings("ignore")

# Configuración de credenciales en Colab / Entorno Local
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key
except Exception as e:
    print(f"Advertencia al cargar API Key de Colab: {e}")

# Verificar si la API Key está presente antes de continuar
if "GOOGLE_API_KEY" not in os.environ:
    raise ValueError("Error: Debes configurar la variable de entorno GOOGLE_API_KEY o añadirla en los Secretos de Colab.")

# Frameworks de Agentes y Estado (IE2)
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from fhir.resources.patient import Patient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

print("=== INICIALIZANDO ENTORNO AUTÓNOMO AGENTE VITALIS (EP2) ===")

# Instanciación del Modelo
NOMBRE_MODELO = "gemini-3.5-flash"
llm = ChatGoogleGenerativeAI(model=NOMBRE_MODELO, temperature=0.0)

# ------------------------------------------------------------------------------
# 1. BASE VECTORIAL Y RAG INTERNO/EXTERNO (CASO CLÍNICA VITALIS)
# ------------------------------------------------------------------------------
def _inicializar_base_vectorial_vitalis():
    """Crea una base FAISS autónoma con datos FHIR (internos) y guías clínicas (externas)."""
    docs_internos = [
        Document(
            page_content="Paciente: Marcela Soto. Genero: female. Fecha de nacimiento: 1991-06-02.",
            metadata={"fuente": "interna", "origen": "HIS_VITALIS", "paciente": "Patient/pac-001"}
        ),
        Document(
            page_content="Condicion Medica: Hipertension arterial esencial. Estado clinico: active. Paciente: Patient/pac-001.",
            metadata={"fuente": "interna", "origen": "HIS_VITALIS", "paciente": "Patient/pac-001"}
        ),
        Document(
            page_content="Observacion clinica (2026-08-20): Presion arterial = 178/112 mmHg. Paciente: Patient/pac-001.",
            metadata={"fuente": "interna", "origen": "HIS_VITALIS", "paciente": "Patient/pac-001"}
        )
    ]
    docs_externos = [
        Document(
            page_content="Guia clinica de Hipertension Arterial: Urgencia hipertensiva es una presion sistolica >= 180 mmHg o diastolica >= 110 mmHg, especialmente con cefalea intensa. Requiere atencion inmediata.",
            metadata={"fuente": "externa", "origen": "Guia_Hipertension_GES"}
        )
    ]

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(docs_internos + docs_externos, embeddings)
    return vectorstore.as_retriever(search_kwargs={"k": 2})

recuperador_rag = _inicializar_base_vectorial_vitalis()

# ------------------------------------------------------------------------------
# 2. HERRAMIENTAS DE CONSULTA Y ESCRITURA
# ------------------------------------------------------------------------------
def tool_consultar_vectorstore_vitalis(consulta: str) -> str:
    """Herramienta de Consulta: Recupera contexto semántico de fichas y guías de la clínica."""
    try:
        docs = recuperador_rag.invoke(consulta)
        if not docs:
            return "Sin resultados en la base de datos de Clínica Vitalis."
        return "\n\n".join(f"[Fuente: {d.metadata.get('fuente')}] {d.page_content}" for d in docs)
    except Exception as e:
        return f"Error en la consulta RAG: {str(e)}"

def tool_escribir_ficha_ehr(paciente_id: str, nota_clinica: str) -> str:
    """Herramienta de Escritura: Formatea el registro clínico en Markdown para visualización limpia."""
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

    salida_markdown = f"""### ✅ **EXITO_EHR: Registro Médico Guardado Correctamente**
* **Origen:** `AGENTE_VITALIS_EP2`
* **Timestamp:** `{timestamp}`
* **ID Paciente:** `{paciente_id}`
* **Estado:** `FIRMADAS_Y_REGISTRADAS`

---

{nota_clinica}
"""
    return salida_markdown

# ------------------------------------------------------------------------------
# 3. CONFIGURACIÓN DE MEMORIA Y ESTADO
# ------------------------------------------------------------------------------
class EpisodicState(TypedDict):
    """Esquema de Estado Episódico (STM) de LangGraph."""
    messages: Annotated[List[Any], operator.add]
    fhir_context: Annotated[List[dict], operator.add]
    semantic_context: str
    plan_steps: List[str]
    current_status: str

# ------------------------------------------------------------------------------
# 4. NODOS DEL GRAFO Y ORQUESTACIÓN DE RAZONAMIENTO
# ------------------------------------------------------------------------------
def node_ingest_fhir(state: EpisodicState):
    """Nodo 1: Ingesta e inspección estricta Pydantic/FHIR para evitar Schema Drift."""
    ultimo_msg = state["messages"][-1].content if state["messages"] else "{}"
    print("\n[NODO INGEST] Validando entrada estocástica contra contrato FHIR...")

    try:
        datos = json.loads(ultimo_msg)
        paciente_fhir = Patient(**datos)
        datos_serializados = paciente_fhir.model_dump(mode="json", exclude_none=True)

        return {
            "fhir_context": [datos_serializados],
            "current_status": "VALIDACION_FHIR_OK",
            "plan_steps": ["consultar_historial", "razonar_conducta", "registrar_ehr"]
        }
    except Exception as e:
        print(f"  └─ Error de contrato FHIR: {e}")
        return {
            "current_status": f"ERROR_VALIDACION_FHIR: {str(e)}",
            "plan_steps": ["abortar"]
        }

def node_retrieve_context(state: EpisodicState):
    """Nodo 2: Invocación de la herramienta de consulta RAG (Memoria Semántica)."""
    print("[NODO RETRIEVE] Invocando herramienta de consulta semántica...")
    fhir_data = state.get("fhir_context", [])

    if not fhir_data:
        consulta = state["messages"][-1].content
    else:
        paciente = fhir_data[-1]
        nombre = paciente.get("name", [{}])[0].get("given", [""])[0]
        consulta = f"Antecedentes clínicos y presión arterial de paciente {nombre}"

    contexto = tool_consultar_vectorstore_vitalis(consulta)
    return {
        "semantic_context": contexto,
        "current_status": "CONTEXTO_RECUPERADO"
    }

def node_reasoning_and_decide(state: EpisodicState):
    """Nodo 3: Motor de Razonamiento Gemini 3.5 Flash para toma de decisiones adaptativas."""
    print("[NODO REASONING] Ejecutando razonamiento clínico adaptativo (Gemini 3.5 Flash)...")

    prompt_sistema = f"""
    Eres el Agente Clínico de Apoyo de Clínica Vitalis.
    Evalúa la información de la memoria episódica y toma una decisión médica.

    MEMORIA DE CONTENIDO (FHIR STM): {json.dumps(state.get('fhir_context', []), ensure_ascii=False)}
    MEMORIA SEMÁNTICA (RAG FAISS): {state.get('semantic_context', '')}
    PLAN ACTUAL: {state.get('plan_steps', [])}

    INSTRUCCIONES DE PLANIFICACIÓN:
    1. Si detectas una crisis/urgencia hipertensiva (PA >= 180/110 mmHg), indica ESCALAMIENTO_URGENTE.
    2. Si los datos no son suficientes para decidir, responde únicamente: 'REQUIERE_MAS_DATOS'.
    3. Si la información es suficiente, genera el resumen clínico consolidado para la ficha médica.
    """

    respuesta_llm = llm.invoke([HumanMessage(content=prompt_sistema)])

    contenido = respuesta_llm.content
    if isinstance(contenido, list):
        texto_limpio = "".join(b.get("text", "") for b in contenido if isinstance(b, dict) and b.get("type") == "text")
    else:
        texto_limpio = str(contenido)

    return {
        "messages": [AIMessage(content=texto_limpio)],
        "current_status": "RAZONAMIENTO_COMPLETADO"
    }

def node_write_ehr_report(state: EpisodicState):
    """Nodo 4: Invocación de la herramienta de escritura (Persistencia en EHR)."""
    print("[NODO WRITE_REPORT] Ejecutando herramienta de escritura en la Ficha Electrónica (EHR)...")
    ultimo_mensaje = state["messages"][-1]

    texto_limpio = ultimo_mensaje.content if hasattr(ultimo_mensaje, "content") else str(ultimo_mensaje)
    fhir_data = state.get("fhir_context", [])
    paciente_id = fhir_data[-1].get("id", "pac-001") if fhir_data else "pac-generico"

    resultado_escritura = tool_escribir_ficha_ehr(paciente_id, texto_limpio)
    return {
        "current_status": resultado_escritura
    }

# ------------------------------------------------------------------------------
# 5. ENRUTAMIENTO CONDICIONAL Y DECISIONES ADAPTATIVAS
# ------------------------------------------------------------------------------
def router_decisiones_clinicas(state: EpisodicState) -> str:
    """Evalúa dinámicamente el estado para dirigir el flujo o abortar ante fallas de contrato."""
    status = state.get("current_status", "")

    if "ERROR_VALIDACION_FHIR" in status:
        print("  └─ [ROUTER] Falla de contrato FHIR detectada. Abortando flujo por seguridad.")
        return END

    ultimo_texto = state["messages"][-1].content if state.get("messages") else ""
    if "REQUIERE_MAS_DATOS" in ultimo_texto:
        print("  └─ [ROUTER] Datos insuficientes. Interrumpiendo ciclo para requerir información.")
        return END

    print("  └─ [ROUTER] Decisión clínica validada. Procediendo a nodo de escritura (EHR).")
    return "write_report"

# ------------------------------------------------------------------------------
# 6. CONSTRUCCIÓN DEL GRAFO DE ESTADO (ORQUESTACIÓN LANGGRAPH)
# ------------------------------------------------------------------------------
workflow_vitalis = StateGraph(EpisodicState)

workflow_vitalis.add_node("ingest_fhir", node_ingest_fhir)
workflow_vitalis.add_node("retrieve_context", node_retrieve_context)
workflow_vitalis.add_node("reasoning", node_reasoning_and_decide)
workflow_vitalis.add_node("write_report", node_write_ehr_report)

workflow_vitalis.add_edge(START, "ingest_fhir")
workflow_vitalis.add_edge("ingest_fhir", "retrieve_context")
workflow_vitalis.add_edge("retrieve_context", "reasoning")

workflow_vitalis.add_conditional_edges(
    "reasoning",
    router_decisiones_clinicas,
    {
        "write_report": "write_report",
        END: END
    }
)

workflow_vitalis.add_edge("write_report", END)

checkpointer = MemorySaver()
agente_vitalis_app = workflow_vitalis.compile(checkpointer=checkpointer)

print("=== GRAFO DE ORQUESTACIÓN COMPILADO Y LISTO ===\n")

# ------------------------------------------------------------------------------
# 7. DEMO EN VIVO
# ------------------------------------------------------------------------------

print("=================================================================")
print(" ESCENARIO 1: PACIENTE VÁLIDO (MARCELA SOTO - FLUJO COMPLETO)")
print("=================================================================")

config_sesion_1 = {"configurable": {"thread_id": "sesion_vitalis_001"}}
input_fhir_valido = json.dumps({
    "resourceType": "Patient",
    "id": "pac-001",
    "name": [{"given": ["Marcela"], "family": "Soto"}],
    "gender": "female",
    "birthDate": "1991-06-02"
})

estado_inicial_1 = {
    "messages": [HumanMessage(content=input_fhir_valido)],
    "fhir_context": [],
    "semantic_context": "",
    "plan_steps": [],
    "current_status": "INICIANDO"
}

for evento in agente_vitalis_app.stream(estado_inicial_1, config_sesion_1):
    for nodo, valores in evento.items():
        print(f" -> Paso completado: [{nodo.upper()}]")
        if nodo == "write_report":
            print("\n[RESULTADO FINAL EHR]\n")
            # Renderizado directo como bloque Markdown estilizado
            display(Markdown(valores.get('current_status', '')))

print("\n=================================================================")
print(" ESCENARIO 2: MANEJO DE ERROR DE CONTRATO (SCHEMA DRIFT)")
print("=================================================================")

config_sesion_2 = {"configurable": {"thread_id": "sesion_vitalis_002"}}
input_fhir_invalido = json.dumps({
    "resourceType": "Patient",
    "id": 99999,  # Viola el esquema Pydantic (debe ser string)
    "gender": "invalid_gender_enum"
})

estado_inicial_2 = {
    "messages": [HumanMessage(content=input_fhir_invalido)],
    "fhir_context": [],
    "semantic_context": "",
    "plan_steps": [],
    "current_status": "INICIANDO"
}

for evento in agente_vitalis_app.stream(estado_inicial_2, config_sesion_2):
    for nodo, valores in evento.items():
        print(f" -> Paso completado: [{nodo.upper()}] | Status: {valores.get('current_status')}")

=== INICIALIZANDO ENTORNO AUTÓNOMO AGENTE VITALIS (EP2) ===


/tmp/ipykernel_3734/1236487434.py:75: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (8 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== GRAFO DE ORQUESTACIÓN COMPILADO Y LISTO ===

 ESCENARIO 1: PACIENTE VÁLIDO (MARCELA SOTO - FLUJO COMPLETO)

[NODO INGEST] Validando entrada estocástica contra contrato FHIR...
 -> Paso completado: [INGEST_FHIR]
[NODO RETRIEVE] Invocando herramienta de consulta semántica...
 -> Paso completado: [RETRIEVE_CONTEXT]
[NODO REASONING] Ejecutando razonamiento clínico adaptativo (Gemini 3.5 Flash)...
  └─ [ROUTER] Decisión clínica validada. Procediendo a nodo de escritura (EHR).
 -> Paso completado: [REASONING]
[NODO WRITE_REPORT] Ejecutando herramienta de escritura en la Ficha Electrónica (EHR)...
 -> Paso completado: [WRITE_REPORT]

[RESULTADO FINAL EHR]



### ✅ **EXITO_EHR: Registro Médico Guardado Correctamente**
* **Origen:** `AGENTE_VITALIS_EP2`
* **Timestamp:** `2026-11-08 15:49:03`
* **ID Paciente:** `pac-001`
* **Estado:** `FIRMADAS_Y_REGISTRADAS`

---

**EVALUACIÓN CLÍNICA Y DECISIÓN MÉDICA**

**Alerta:** **ESCALAMIENTO_URGENTE**

---

### **Resumen Clínico Consolidado (Ficha Médica - EHR)**

* **Paciente:** Marcela Soto
* **ID:** `pac-001`
* **Género:** Femenino
* **Fecha de Nacimiento:** 02/06/1991
* **Diagnóstico Activo:** Hipertensión arterial esencial (Condition: Active)
* **Última Observación Clínica (2026-08-20):** Presión Arterial = **178/112 mmHg**

---

### **Conducta y Plan de Acción**
1. **Clasificación:** El registro de Presión Arterial (TAD = 112 mmHg $\ge$ 110 mmHg) cumple con los criterios de **Crisis / Urgencia Hipertensiva**.
2. **Acción Inmediata:** Activar **ESCALAMIENTO_URGENTE** para derivación o evaluación médica presencial/inmediata, descarte de daño a órgano blanco y ajuste de terapia farmacológica antihipertensiva.
3. **Registro:** Almacenar el presente resumen y alerta en el EHR del paciente (`pac-001`).



 ESCENARIO 2: MANEJO DE ERROR DE CONTRATO (SCHEMA DRIFT)

[NODO INGEST] Validando entrada estocástica contra contrato FHIR...
  └─ Error de contrato FHIR: 1 validation error for Patient
id
  Input should be a valid string [type=string_type, input_value=99999, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
 -> Paso completado: [INGEST_FHIR] | Status: ERROR_VALIDACION_FHIR: 1 validation error for Patient
id
  Input should be a valid string [type=string_type, input_value=99999, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
[NODO RETRIEVE] Invocando herramienta de consulta semántica...
 -> Paso completado: [RETRIEVE_CONTEXT] | Status: CONTEXTO_RECUPERADO
[NODO REASONING] Ejecutando razonamiento clínico adaptativo (Gemini 3.5 Flash)...
  └─ [ROUTER] Datos insuficientes. Interrumpiendo ciclo para requerir información.
 -> Paso completado: [REASONING] | Status: RAZONAMIENTO_COMPLETADO
